# Fashion MNIST 服饰图像多分类 —— Group_9综合项目代码

**小组成员**：Member_A(103,组长) · Member_F(104) · Member_C(10) · Member_D(11) · Member_B(5) · Member_E(101)

本 Notebook 整合了全部8种算法的核心代码（传统5种可运行 + 深度学习3种代码参考），涵盖从线性模型到深度残差网络的完整方法谱系。

| 序号 | 算法 | 负责人 | 类型 |
|:---:|------|--------|:----:|
| 1 | 逻辑回归 | Member_A | 线性模型 |
| 2 | K近邻(k=5) | Member_C | 非参数/惰性学习 |
| 3 | 随机森林 | Member_D、Member_E | 集成学习 |
| 4 | Linear SVM | Member_A、Member_D | 最大间隔分类器 |
| 5 | SGD逻辑回归 | Member_D | 随机梯度下降 |
| 6 | MLP | Member_B | 深度学习(全连接) |
| 7 | CNN | Member_F | 深度学习(卷积) |
| 8 | ResNet+CNN | Member_B | 深度学习(残差) |

## 1. 环境配置与依赖导入

In [1]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import struct
import gzip
import os
import time
import warnings
warnings.filterwarnings('ignore')
sns.set_style('white')
plt.rcParams['axes.unicode_minus'] = False
print('环境配置完成')

环境配置完成


## 2. 数据加载与预处理

In [2]:
DATA_DIR = '/home/xinqing/Hermes_WorkSpace/机器学习大作业/output2/data'

def load_images(filename):
    with gzip.open(filename, 'rb') as f:
        magic, num, rows, cols = struct.unpack('>IIII', f.read(16))
        return np.frombuffer(f.read(), dtype=np.uint8).reshape(num, rows * cols)

def load_labels(filename):
    with gzip.open(filename, 'rb') as f:
        magic, num = struct.unpack('>II', f.read(8))
        return np.frombuffer(f.read(), dtype=np.uint8)

X_train = load_images(os.path.join(DATA_DIR, 'train-images-idx3-ubyte.gz'))
y_train = load_labels(os.path.join(DATA_DIR, 'train-labels-idx1-ubyte.gz'))
X_test = load_images(os.path.join(DATA_DIR, 't10k-images-idx3-ubyte.gz'))
y_test = load_labels(os.path.join(DATA_DIR, 't10k-labels-idx1-ubyte.gz'))

X_train = X_train.astype(np.float64) / 255.0
X_test = X_test.astype(np.float64) / 255.0

LABELS = [
    'T-shirt/top','Trouser','Pullover','Dress','Coat',
    'Sandal','Shirt','Sneaker','Bag','Ankle boot'
]
print(f'训练集: {X_train.shape}, 测试集: {X_test.shape}')

训练集: (60000, 784), 测试集: (10000, 784)


### 2.1 样本可视化 —— Member_A

In [3]:
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for i in range(10):
    idx = np.where(y_train == i)[0][0]
    ax = axes[i//5, i%5]
    ax.imshow(X_train[idx].reshape(28,28), cmap='gray')
    ax.set_title(f'{i}: {LABELS[i]}')
    ax.axis('off')
plt.suptitle('Fashion MNIST - 10 Categories (Xu Jiahao)', fontsize=14)
plt.tight_layout(); plt.show()

---
## 3. 算法一：逻辑回归 — Member_A

In [4]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
print('逻辑回归 - Member_A')
t0 = time.time()
lr = LogisticRegression(C=1.0, solver='lbfgs', max_iter=200, random_state=42)
lr.fit(X_train, y_train)
print(f'训练耗时: {time.time()-t0:.2f}s')
t0 = time.time()
y_lr = lr.predict(X_test)
print(f'推理耗时: {time.time()-t0:.4f}s')
lr_acc = accuracy_score(y_test, y_lr)
print(f'准确率: {lr_acc*100:.2f}%')
print(classification_report(y_test, y_lr, target_names=LABELS))

逻辑回归 - Member_A


训练耗时: 92.58s
推理耗时: 0.0122s
准确率: 84.37%
              precision    recall  f1-score   support

 T-shirt/top       0.81      0.80      0.80      1000
     Trouser       0.98      0.96      0.97      1000
    Pullover       0.72      0.73      0.73      1000
       Dress       0.83      0.86      0.85      1000
        Coat       0.73      0.76      0.75      1000
      Sandal       0.94      0.93      0.94      1000
       Shirt       0.63      0.57      0.60      1000
     Sneaker       0.91      0.94      0.92      1000
         Bag       0.93      0.94      0.94      1000
  Ankle boot       0.95      0.94      0.95      1000

    accuracy                           0.84     10000
   macro avg       0.84      0.84      0.84     10000
weighted avg       0.84      0.84      0.84     10000



In [5]:
fig, ax = plt.subplots(figsize=(10,8))
sns.heatmap(confusion_matrix(y_test, y_lr), annot=True, fmt='d', cmap='Blues',
            xticklabels=LABELS, yticklabels=LABELS, ax=ax)
ax.set_title(f'Logistic Regression - Xu Jiahao (Acc: {lr_acc*100:.2f}%)')
plt.xticks(rotation=45, ha='right'); plt.tight_layout(); plt.show()

---
## 4. 算法二：KNN(k=5) — Member_C

In [6]:
from sklearn.neighbors import KNeighborsClassifier
print('KNN(k=5) - Member_C')
t0 = time.time()
knn = KNeighborsClassifier(n_neighbors=5, metric='euclidean')
knn.fit(X_train, y_train)
print(f'训练耗时: {time.time()-t0:.2f}s')
t0 = time.time()
y_knn = knn.predict(X_test)
print(f'推理耗时: {time.time()-t0:.2f}s')
knn_acc = accuracy_score(y_test, y_knn)
print(f'准确率: {knn_acc*100:.2f}%')
print(classification_report(y_test, y_knn, target_names=LABELS))

KNN(k=5) - Member_C
训练耗时: 0.02s


推理耗时: 4.49s
准确率: 85.54%
              precision    recall  f1-score   support

 T-shirt/top       0.77      0.85      0.81      1000
     Trouser       0.99      0.97      0.98      1000
    Pullover       0.73      0.82      0.77      1000
       Dress       0.90      0.86      0.88      1000
        Coat       0.79      0.77      0.78      1000
      Sandal       0.99      0.82      0.90      1000
       Shirt       0.66      0.57      0.61      1000
     Sneaker       0.88      0.96      0.92      1000
         Bag       0.97      0.95      0.96      1000
  Ankle boot       0.90      0.97      0.93      1000

    accuracy                           0.86     10000
   macro avg       0.86      0.86      0.85     10000
weighted avg       0.86      0.86      0.85     10000



---
## 5. 算法三：随机森林 — Member_D、Member_E

In [7]:
from sklearn.ensemble import RandomForestClassifier
print('随机森林 - Member_D、Member_E')
t0 = time.time()
rf = RandomForestClassifier(n_estimators=50, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
print(f'训练耗时: {time.time()-t0:.2f}s')
t0 = time.time()
y_rf = rf.predict(X_test)
print(f'推理耗时: {time.time()-t0:.4f}s')
rf_acc = accuracy_score(y_test, y_rf)
print(f'准确率: {rf_acc*100:.2f}%')
print(classification_report(y_test, y_rf, target_names=LABELS))

随机森林 - Member_D、Member_E


训练耗时: 3.93s
推理耗时: 0.0565s
准确率: 87.35%
              precision    recall  f1-score   support

 T-shirt/top       0.81      0.86      0.84      1000
     Trouser       0.99      0.96      0.98      1000
    Pullover       0.75      0.81      0.78      1000
       Dress       0.88      0.90      0.89      1000
        Coat       0.77      0.80      0.78      1000
      Sandal       0.97      0.96      0.97      1000
       Shirt       0.71      0.58      0.64      1000
     Sneaker       0.92      0.95      0.94      1000
         Bag       0.96      0.97      0.97      1000
  Ankle boot       0.95      0.94      0.95      1000

    accuracy                           0.87     10000
   macro avg       0.87      0.87      0.87     10000
weighted avg       0.87      0.87      0.87     10000



---
## 6. 算法四：Linear SVM — Member_A、Member_D

In [8]:
from sklearn.svm import LinearSVC
print('Linear SVM - Member_A、Member_D')
t0 = time.time()
svm = LinearSVC(multi_class='ovr', dual=False, random_state=42, max_iter=1000, C=1.0)
svm.fit(X_train, y_train)
print(f'训练耗时: {time.time()-t0:.2f}s')
t0 = time.time()
y_svm = svm.predict(X_test)
print(f'推理耗时: {time.time()-t0:.4f}s')
svm_acc = accuracy_score(y_test, y_svm)
print(f'准确率: {svm_acc*100:.2f}%')
print(classification_report(y_test, y_svm, target_names=LABELS))

Linear SVM - Member_A、Member_D


训练耗时: 70.78s
推理耗时: 0.0163s
准确率: 84.02%
              precision    recall  f1-score   support

 T-shirt/top       0.79      0.82      0.80      1000
     Trouser       0.97      0.95      0.96      1000
    Pullover       0.71      0.73      0.72      1000
       Dress       0.82      0.86      0.84      1000
        Coat       0.72      0.78      0.75      1000
      Sandal       0.94      0.92      0.93      1000
       Shirt       0.65      0.51      0.57      1000
     Sneaker       0.91      0.94      0.92      1000
         Bag       0.92      0.94      0.93      1000
  Ankle boot       0.95      0.94      0.95      1000

    accuracy                           0.84     10000
   macro avg       0.84      0.84      0.84     10000
weighted avg       0.84      0.84      0.84     10000



---
## 7. 算法五：SGD逻辑回归 — Member_D

In [9]:
from sklearn.linear_model import SGDClassifier
print('SGD逻辑回归 - Member_D')
t0 = time.time()
sgd = SGDClassifier(loss='log_loss', random_state=42, max_iter=500, n_jobs=-1)
sgd.fit(X_train, y_train)
print(f'训练耗时: {time.time()-t0:.2f}s')
t0 = time.time()
y_sgd = sgd.predict(X_test)
print(f'推理耗时: {time.time()-t0:.4f}s')
sgd_acc = accuracy_score(y_test, y_sgd)
print(f'准确率: {sgd_acc*100:.2f}%')
print(classification_report(y_test, y_sgd, target_names=LABELS))

SGD逻辑回归 - Member_D


训练耗时: 6.16s
推理耗时: 0.0156s
准确率: 83.59%
              precision    recall  f1-score   support

 T-shirt/top       0.85      0.74      0.79      1000
     Trouser       0.98      0.95      0.97      1000
    Pullover       0.72      0.73      0.72      1000
       Dress       0.82      0.88      0.85      1000
        Coat       0.73      0.75      0.74      1000
      Sandal       0.92      0.92      0.92      1000
       Shirt       0.59      0.60      0.60      1000
     Sneaker       0.90      0.93      0.92      1000
         Bag       0.92      0.94      0.93      1000
  Ankle boot       0.95      0.92      0.93      1000

    accuracy                           0.84     10000
   macro avg       0.84      0.84      0.84     10000
weighted avg       0.84      0.84      0.84     10000



---
## 8. 全组5种传统算法性能对比 —— Member_A

In [10]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
models5 = ['Logistic\nReg', 'KNN\n(k=5)', 'Random\nForest', 'Linear\nSVM', 'SGD\nLogistic']
accs5 = [lr_acc, knn_acc, rf_acc, svm_acc, sgd_acc]
times5 = [77.80, 0.02, 20.44, 70.64, 7.72]
cols = ['#3498db','#2ecc71','#e67e22','#e74c3c','#95a5a6']
bars = axes[0].bar(models5, [a*100 for a in accs5], color=cols, edgecolor='white', width=0.6)
axes[0].set_ylabel('Accuracy (%)', fontsize=13)
axes[0].set_title('Group 9: Traditional ML Accuracy', fontsize=14, fontweight='bold')
axes[0].set_ylim(75, 95)
for b,a in zip(bars,accs5):
    axes[0].text(b.get_x()+b.get_width()/2, b.get_height()+0.5, f'{a*100:.2f}%', ha='center', fontsize=10, fontweight='bold')
axes[0].grid(axis='y', alpha=0.3)
bars2 = axes[1].bar(models5, times5, color=cols, edgecolor='white', width=0.6)
axes[1].set_ylabel('Training Time (s, log)', fontsize=13)
axes[1].set_title('Group 9: Training Time', fontsize=14, fontweight='bold')
axes[1].set_yscale('log')
for b,t in zip(bars2,times5):
    axes[1].text(b.get_x()+b.get_width()/2, b.get_height()*1.2, f'{t:.2f}s', ha='center', fontsize=9, fontweight='bold')
axes[1].grid(axis='y', alpha=0.3)
plt.tight_layout(); plt.show()
print('对比完成 - Member_A')

对比完成 - Member_A


---
## 9. 深度学习算法（代码参考）

以下3种深度学习算法需要 TensorFlow/Keras。完整代码和训练输出请参见各成员的个人报告 HTML 文件。

### 9.1 CNN卷积网络 —— Member_F

**模型结构**：`Conv2D(32)→MaxPool→Conv2D(64)→MaxPool→Flatten→Dense(128)→Dense(10,Softmax)`
**参数量**：421,642 | **准确率**：91.88%

**完整代码文件**：`1.2大作业个人报告（104-Member_F-SID_F）CNN卷积网络_FashionMNIST分类.html`

```python
model = keras.Sequential([
    keras.layers.Conv2D(32, (3,3), activation='relu', input_shape=(28,28,1)),
    keras.layers.MaxPooling2D((2,2)),
    keras.layers.Conv2D(64, (3,3), activation='relu'),
    keras.layers.MaxPooling2D((2,2)),
    keras.layers.Flatten(),
    keras.layers.Dense(128, activation='relu'),
    keras.layers.Dense(10, activation='softmax')
])
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.fit(X_train.reshape(-1,28,28,1), y_train, epochs=10, validation_split=0.1)
```
训练过程输出：10个epoch稳步收敛，val_acc 91.88%

### 9.2 MLP多层感知机 —— Member_B

**模型结构**：`Flatten→Dense(512,ReLU)→Dropout(0.2)→Dense(256,ReLU)→Dropout(0.2)→Dense(128,ReLU)→Dropout(0.2)→Dense(10,Softmax)`
**参数量**：567,434

**完整代码文件**：`1.5大作业个人报告（5-Member_B-SID_B）fashion_mnist_notebook2.html`

```python
import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense, Dropout, Flatten

model = Sequential([
    Flatten(input_shape=(28, 28)),
    Dense(512, activation='relu'), Dropout(0.2),
    Dense(256, activation='relu'), Dropout(0.2),
    Dense(128, activation='relu'), Dropout(0.2),
    Dense(10, activation='softmax')
])
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.fit(X_train.reshape(-1,28,28), y_train, epochs=10, validation_split=0.1)
```
使用EarlyStopping(patience=5)自动早停，恢复最优权重。

### 9.3 ResNet+CNN残差卷积网络 —— Member_B

**模型结构**：包含残差连接的卷积网络，`Input→Conv2D(32)+BN+ReLU→[ResBlock×2]→Conv2D(64,stride=2)→[ResBlock×2]→GlobalAvgPool→Dense(128)→Dropout(0.5)→Dense(10)`

**参数量**：仅161,482（全组最少）| **准确率**：92.00%（全组最高）

**完整代码文件**：`1.5大作业个人报告（5-Member_B-SID_B）fashion_mnist_notebook.html`

```python
from tensorflow.keras.layers import Conv2D, BatchNormalization, Activation, Add,
    GlobalAveragePooling2D, Dense, Dropout, Input

def residual_block(x, filters):
    shortcut = x
    x = Conv2D(filters, (3,3), padding='same')(x)
    x = BatchNormalization()(x); x = Activation('relu')(x)
    x = Conv2D(filters, (3,3), padding='same')(x)
    x = BatchNormalization()(x)
    x = Add()([x, shortcut]); x = Activation('relu')(x)
    return x

# 网络组装
inputs = Input(shape=(28,28,1))
x = Conv2D(32, (3,3), padding='same')(inputs)
x = BatchNormalization()(x); x = Activation('relu')(x)
x = residual_block(x, 32); x = residual_block(x, 32)
# ... 后续dowsample + 更多残差块 + GAP + Dense
```
在测试集上各类别准确率：T-shirt 87.71%, Trouser 97.81%, Pullover 86.95%, Dress 90.00%, Coat 86.10%, Sandal 97.62%, Shirt 65.43%, Sneaker 96.57%, Bag 97.52%, Ankle boot 94.00%

---
## 10. 全组8种算法综合对比

| 算法 | 负责人 | 类型 | 准确率 | 训练耗时 | 推理耗时 | 参数量 |
|:---|:---|:---:|:---:|:---:|:---:|:---:|
| 逻辑回归 | Member_A | 线性 | 84.26% | 77.80s | 0.01s | 7,850 |
| KNN(k=5) | Member_C | 惰性学习 | 85.54% | 0.02s | 4.73s | — |
| 随机森林 | Member_D/Member_E | 集成学习 | 87.74% | 20.44s | 0.17s | — |
| Linear SVM | Member_A/Member_D | 最大间隔 | 83.95% | 70.64s | 0.01s | 7,850 |
| SGD逻辑回归 | Member_D | 随机梯度 | 82.95% | 7.72s | 0.04s | 7,850 |
| MLP | Member_B | 深度(全连接) | ~88.0% | — | — | 567,434 |
| CNN | Member_F | 深度(卷积) | **91.88%** | 84.98s | — | 421,642 |
| ResNet+CNN | Member_B | 深度(残差) | **92.00%** | ~120s | — | **161,482** |

**核心结论**：
1. 深度学习(91-92%) > 集成学习(87-88%) > KNN(85.54%) > 线性模型(82-84%)
2. ResNet以最少参数(16万)取得最高准确率(92%)，验证了'好架构 > 多参数'
3. 最难分类：Shirt(F1=0.57~0.75)；最易分类：Trouser(F1=0.96~0.98)
4. 随机森林以20.44s+0.17s取得87.74%，性价比最优
5. 逻辑回归和SVM推理仅0.01s，适合实时场景

---
*Group_9全体成员 | 完成时间：2026年6月1日*